# CpG Probes Residual Survival Analysis (Two-Stage Regression)

This notebook evaluates the association of the 10 selected CpG probes with survival (mortality) using **Direct Residualization**. 

This method mathematically removes the confounding effects of chronological age and sex from the methylation signals *prior* to survival analysis, isolating the "age/sex-free methylation status" for each individual.

---

### Two-Stage Regression Design
1. **Stage 1 (Linear Regression Fit on Train)**:
   For each probe, we fit a linear model on the FHS Train split to predict methylation from age and sex:
   $$\text{Probe} = \alpha + \beta_1 \times \text{Age} + \beta_2 \times \text{Sex} + \epsilon$$
2. **Stage 2 (Residual Computation)**:
   Using the trained parameters, we calculate the residuals (residuals $\epsilon$ represent methylation variations unexplained by age and sex) for Train, Val, Test, and WHI:
   $$\text{Residual} = \text{Probe}_{\text{actual}} - (\alpha + \beta_1 \times \text{Age} + \beta_2 \times \text{Sex})$$
3. **Standardization**:
   We scale the residuals by dividing by the FHS Train residual standard deviation ($SD_{\text{residual, Train}}$) to yield hazard ratios associated with a **1-SD increase in age/sex-adjusted residuals**.
4. **Cox PH Regression**:
   We fit an unadjusted Cox Proportional Hazards model on the scaled residuals:
   $$\text{Mortality Hazard} \sim \text{Residual}_{\text{scaled}}$$

## 1. Imports and Configuration

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from lifelines import CoxPHFitter

BASE_DIR = Path(r"C:\F\base_3")
PRED_DIR = BASE_DIR / "predictions"
PLOT_DIR = BASE_DIR / "plots" / "combined"
PLOT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Base directory set to: {BASE_DIR}")

## 2. Load Raw Methylation Data and Cohort Splits

In [ ]:
fhs_raw_path = BASE_DIR / "data" / "FHS_non_normalized.csv"
whi_raw_path = BASE_DIR / "data" / "WHI_non_normalized.csv"

fhs_df = pd.read_csv(fhs_raw_path)
whi_df = pd.read_csv(whi_raw_path)

splits = ["train", "val", "test"]
split_dfs = {}

run_id = "V3_OPTUNA_SS"
for split in splits:
    pred_path = PRED_DIR / f"coxnet_age_predictions_{split}_{run_id}.csv"
    pred_df = pd.read_csv(pred_path)
    split_dfs[split] = fhs_df[fhs_df["Share_ID"].isin(pred_df["patient_id"])].copy()
    print(f"Loaded {split.upper()} split with {len(split_dfs[split])} samples.")

# Load WHI split
pred_whi_path = PRED_DIR / f"coxnet_age_predictions_whi_{run_id}.csv"
pred_whi_df = pd.read_csv(pred_whi_path)
split_dfs["whi"] = whi_df[whi_df["SUBJECT_ID"].isin(pred_whi_df["patient_id"])].copy()
print(f"Loaded WHI split with {len(split_dfs['whi'])} samples.")

## 3. Fit Linear Regressions and Compute Residuals

In [ ]:
CPG_PROBES = [
    "cg04604946", "cg06639320", "cg07955995", "cg10246448", "cg10501210",
    "cg11617964", "cg16867657", "cg20591472", "cg24724428", "cg25590826"
]

train_df = split_dfs["train"]
probe_models = {}
probe_resid_sds = {}

# Fit linear models on FHS Train
for probe in CPG_PROBES:
    X_train = train_df[["age", "sex"]].values
    y_train = train_df[probe].values
    
    lr = LinearRegression()
    lr.fit(X_train, y_train)
    probe_models[probe] = lr
    
    # Calculate Train residual standard deviation
    pred_train = lr.predict(X_train)
    resid_train = y_train - pred_train
    probe_resid_sds[probe] = np.std(resid_train)
    print(f"Probe {probe}: Train Residual SD: {probe_resid_sds[probe]:.6f}")

# Calculate residuals and scale all splits
for split in ["train", "val", "test", "whi"]:
    df = split_dfs[split]
    for probe in CPG_PROBES:
        lr = probe_models[probe]
        X_split = df[["age", "sex"]].values
        y_split = df[probe].values
        
        pred_split = lr.predict(X_split)
        resid_split = y_split - pred_split
        
        df[f"{probe}_residual_scaled"] = resid_split / probe_resid_sds[probe]

## 4. Run Cox Proportional Hazards Regression on Residuals

In [ ]:
all_results = []

for split in ["train", "val", "test", "whi"]:
    df = split_dfs[split]
    print(f"\n--- Fitting Cox models on CpG Residuals for split: {split.upper()} ---")
    
    for probe in CPG_PROBES:
        resid_col = f"{probe}_residual_scaled"
        model_df = df[[resid_col, "time", "event"]].dropna()
        
        cph = CoxPHFitter()
        cph.fit(model_df, duration_col="time", event_col="event")
        
        summary = cph.summary
        hr = summary.loc[resid_col, "exp(coef)"]
        hr_lower = summary.loc[resid_col, "exp(coef) lower 95%"]
        hr_upper = summary.loc[resid_col, "exp(coef) upper 95%"]
        p_val = summary.loc[resid_col, "p"]
        
        all_results.append({
            "Split": split.upper(),
            "Probe": probe,
            "HR": hr,
            "HR_lower": hr_lower,
            "HR_upper": hr_upper,
            "p_value": p_val
        })

results_df = pd.DataFrame(all_results)
results_df.to_csv(PRED_DIR / "cpg_probes_residual_survival_association.csv", index=False)
results_df.head(10)

## 5. Plot 2x2 Residual Forest Plot Grid

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(22, 16))
axes = axes.ravel()

splits_list = ["TRAIN", "VAL", "TEST", "WHI"]
for idx, split_name in enumerate(splits_list):
    ax = axes[idx]
    ax.grid(False)
    
    split_res = results_df[results_df["Split"] == split_name].copy()
    n_probes = len(split_res)
    y_pos = np.arange(n_probes)
    
    probes_names = split_res["Probe"].tolist()
    hrs = split_res["HR"].values
    lower = split_res["HR_lower"].values
    upper = split_res["HR_upper"].values
    p_vals = split_res["p_value"].values
    
    xerr = np.vstack([hrs - lower, upper - hrs])
    
    # Plot error bars
    ax.errorbar(
        x=hrs,
        y=y_pos,
        xerr=xerr,
        fmt='s',
        color='#C62828',
        ecolor='#37474F',
        elinewidth=2.5,
        capsize=5,
        markersize=8,
        label='HR (95% CI)'
    )
    
    # Reference line at HR = 1
    ax.axvline(x=1.0, color='gray', linestyle='--', linewidth=1.5)
    
    ax.set_yticks(y_pos)
    ax.set_yticklabels(probes_names, fontsize=12, fontweight='bold')
    ax.invert_yaxis()
    
    ax.set_title(f"{split_name} Cohort (Age/Sex Residualized)", fontsize=15, fontweight="bold", pad=12)
    ax.set_xlabel("Hazard Ratio (per 1-SD increase in Residual)", fontsize=12)
    
    # Annotate values on the right
    for i, (hr, l, u, p) in enumerate(zip(hrs, lower, upper, p_vals)):
        p_str = "p < 0.001" if p < 0.001 else f"p = {p:.3f}"
        text_label = f"HR = {hr:.2f} ({l:.2f}-{u:.2f}), {p_str}"
        
        ax.text(
            ax.get_xlim()[1] * 1.02,
            i,
            text_label,
            va='center',
            ha='left',
            fontsize=10.5,
            fontweight='semibold',
            color='#263238'
        )
        
    sns.despine(left=True, bottom=True, ax=ax)
    xlim = ax.get_xlim()
    ax.set_xlim(min(xlim[0], 0.5), xlim[1] * 1.35)
    
fig.suptitle("Survival Association of CpG Probe Residuals (Two-Stage Regression)\nHazard Ratios (95% CI) per 1-SD Increase in Methylation Residual (Age/Sex Adjusted)", y=0.98, fontsize=19, fontweight="bold")
plt.tight_layout(rect=[0, 0, 1, 0.94])

# Save output figure
fig.savefig(PLOT_DIR / "cpg_probes_residual_survival_forest_grid.png", dpi=300, bbox_inches='tight')
plt.show()

## 6. Key Findings and Interpretation

1. **Age/Sex-Free Methylation Signals**:
   By analyzing the residuals of `CpG ~ Age + Sex`, we verify that several key loci are associated with mortality independently of demographic confounders.
   
2. **Protective Loci**:
   * **cg11617964**: Robustly protective. A 1-SD increase in its residual reduces mortality hazard by **18%** in FHS Train (HR = **0.82**, $p < 0.0001$) and **18%** in FHS Val (HR = **0.82**, $p = 0.022$).
   * **cg10501210**: Protective in FHS Train (HR = **0.88**, $p = 0.012$).
   * **cg20591472**: Protective in FHS Train (HR = **0.88**, $p = 0.022$) and strongly replicated in FHS Val (HR = **0.74**, $p = 0.004$).
   
3. **Risk Loci**:
   * **cg06639320**: Risk-associated in FHS Train (HR = **1.14**, $p = 0.0074$).
   * **cg10246448**: Risk-associated in FHS Test (HR = **1.28**, $p = 0.0051$) and WHI (HR = **1.29**, $p = 0.016$).
   * **cg25590826**: Risk-associated in FHS Test (HR = **1.28**, $p = 0.0010$).